# نظام متكامل لكشف وقراءة لوحات السيارات المصرية من الفيديو (YOLO26 + Character-Detection OCR)

## القرارات اللي هبني عليها الحل، وليه

### ١. أي نسخة YOLO؟ → **YOLO26**
- طرحتها Ultralytics في يناير 2026، وهي دلوقتي **النسخة الافتراضية الموصى بيها لأي مشروع جديد**
  (رسمياً في توثيق Ultralytics: "YOLO26 is the recommended starting point for new Ultralytics projects").
- بتشيل NMS تماماً (end-to-end inference) → استنتاج أسرع على الفيديو، وده مهم جداً هنا لأننا هنشغّل موديلين
  (كشف اللوحة + كشف الحروف) على كل فريم.
- محسّنة خصيصاً لـ **الأجسام الصغيرة** (ProgLoss + STAL) — وده أهم سبب عملي ليها هنا، لأن الحروف والأرقام
  جوه اللوحة صغيرة جداً بالنسبة لحجم الصورة، وده بالظبط نوع المشكلة اللي YOLO26 اتحسّنت فيها عن YOLO11.
- لسه بتدعم YOLOv12/YOLO11 نفس الـ API بالظبط، فلو حبيت تجرب واحد منهم بدالها، غيّر اسم الـ checkpoint بس
  (`yolo26s.pt` → `yolo12s.pt` أو `yolo11s.pt`) والباقي هيشتغل زي ما هو.

### ٢. أي طريقة OCR؟ → **كشف كل حرف/رقم كـ object detection (YOLO)، مش CRNN ولا EasyOCR/Tesseract**
الأسباب:
1. **الأدوات الجاهزة (Tesseract, EasyOCR) بتفشل مع اللوحات العربية** — تقريباً كل الأبحاث الحديثة عن اللوحات
   المصرية/العربية بتقول نفس الكلام: النص العربي المطبوع بخط اللوحات + اتجاه الأرقام بيلخبط الموديلات العامة.
2. **الداتا اللي هتستخدمها (Roboflow: `egyptian-car-plates`) اتعملها annotation على مستوى الحرف مش على
   مستوى اللوحة كلها** (38 كلاس: 10 أرقام + 28 حرف عربي) ووصلت **98.8% mAP@50** بموديل YOLOv8s بسيط.
   يعني الداتا نفسها مبنية بالظبط لنموذج "كشف حروف"، مش لموديل sequence-to-sequence.
3. كشف الحرف كـ object detection **مايحتاجش محاذاة/تقطيع للحروف** (زي الـ CNN classifier التقليدي) ولا
   CTC decoding (زي CRNN) — كل حرف بيطلع بصندوق وكلاس وثقة (confidence) لوحده، وده أسهل تصحيح وتشخيص
   للأخطاء لاحقاً (تقدر تعرف بالظبط أنهي حرف اتلخبط مع أنهي).

### ٣. البنية العامة: **Two-Stage Pipeline**
```
فيديو ──▶ Stage 1: YOLO26 (كشف + تتبّع اللوحة كـ object واحد) ──▶ قصّ اللوحة من كل فريم
                                                                        │
                                                                        ▼
                                          Stage 2: YOLO26 (كشف كل حرف/رقم جوه اللوحة المقصوصة)
                                                                        │
                                                                        ▼
                                رتّب الحروف حسب موضعها الأفقي (x) ← كوّن نص اللوحة
                                                                        │
                                                                        ▼
                        صوّت (majority vote) على كل قراءات نفس اللوحة عبر كذا فريم ← قراءة نهائية مستقرة
```

### الداتاسِتات المستخدمة
- **Stage 1 (كشف اللوحة في الصورة الكاملة):**
  `roboflow-universe-projects/license-plate-recognition-rxg4e` — داتاسِت عام لكشف اللوحات (كلاس واحد
  `license-plate`)، 10,125 صورة، منتشر جداً ومستخدم في عشرات المشاريع. مش مصري تحديداً، لكن مهمته بس
  "فين اللوحة في الصورة" — وده مش مرتبط باللغة، فبيشتغل كويس مع لوحات مصر برضه.
- **Stage 2 (كشف الحروف/الأرقام جوه اللوحة):**
  `alyalsayed-vyx6g/egyptian-car-plates` (اللي بعتّه) — 3,725 صورة للوحات مصرية مقصوصة، 38 كلاس
  (10 أرقام + 28 حرف عربي)، موديل مرجعي عليها وصل لـ 98.8% mAP@50 و 99% recall.

> **تنبيه مهم:** لازم يكون عندك حساب Roboflow (مجاني) وAPI key عشان تنزّل الداتاسِتين برمجياً.
> من https://app.roboflow.com/settings/api


## 0. التثبيت

In [ ]:
#@title تثبيت المكتبات المطلوبة
# ultralytics: بيه YOLO26/YOLOv12/YOLO11 كلهم بنفس الـ API
# roboflow: عشان ننزّل الداتاسِتين بصيغة YOLO جاهزة (صور + labels + data.yaml)
# arabic-reshaper + python-bidi: عشان نرسم نص عربي صح فوق فريمات الفيديو (OpenCV وحده مايعرفش يشكّل عربي)
!pip -q install ultralytics roboflow arabic-reshaper python-bidi opencv-python-headless

import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "لا يوجد — روح لـ Runtime > Change runtime type > GPU")


In [ ]:
#@title مفتاح Roboflow API
# حط مفتاحك هنا (من https://app.roboflow.com/settings/api) — أو استخدم Colab Secrets (أأمن)
ROBOFLOW_API_KEY = "YOUR_ROBOFLOW_API_KEY"  # TODO: غيّرها

from roboflow import Roboflow
rf = Roboflow(api_key=ROBOFLOW_API_KEY)


## 1. تنزيل داتا Stage 1 — كشف اللوحة في الصورة الكاملة

In [ ]:
#@title تنزيل داتاسِت كشف اللوحات العام (single class: license-plate)
# نزّلها بصيغة YOLO26 (نفس فورمات YOLOv8/v11 — txt labels + data.yaml)
project_stage1 = rf.workspace("roboflow-universe-projects").project("license-plate-recognition-rxg4e")
# لو الفيرجن رقم 11 مش موجود وقت ما تشغّل الكود، افتح صفحة الداتاسِت في المتصفح وشوف آخر رقم فيرجن وحدّثه هنا
version_stage1 = project_stage1.version(11)
dataset_stage1 = version_stage1.download("yolov11")  # فورمات التصدير متوافق مع yolo26 برضه (نفس بنية labels)

print("Stage 1 dataset saved at:", dataset_stage1.location)
!cat {dataset_stage1.location}/data.yaml


## 2. تنزيل داتا Stage 2 — كشف الحروف/الأرقام جوه اللوحة المصرية

In [ ]:
#@title تنزيل داتاسِت egyptian-car-plates (38 كلاس: أرقام + حروف عربية)
project_stage2 = rf.workspace("alyalsayed-vyx6g").project("egyptian-car-plates")
# تأكد من رقم آخر فيرجن من صفحة الداتاسِت (تبويب "Dataset") قبل التشغيل، وحدّث الرقم هنا لو مختلف
version_stage2 = project_stage2.version(3)
dataset_stage2 = version_stage2.download("yolov11")

print("Stage 2 dataset saved at:", dataset_stage2.location)
!cat {dataset_stage2.location}/data.yaml


In [ ]:
#@title اقرأ أسماء الكلاسات الحقيقية من data.yaml (المصدر الوحيد اللي نثق فيه)
import yaml

with open(f"{dataset_stage2.location}/data.yaml", encoding="utf-8") as f:
    stage2_yaml = yaml.safe_load(f)

STAGE2_CLASS_NAMES = stage2_yaml["names"]
print(f"عدد الكلاسات: {len(STAGE2_CLASS_NAMES)}")
print(STAGE2_CLASS_NAMES)


## 3. تدريب Stage 1 — موديل كشف اللوحة (YOLO26)

- كلاس واحد بس (`license-plate`)، فمش محتاجين موديل كبير — `yolo26s` (small) هيوازن بين السرعة والدقة كويس
  لأنه هيشتغل على فيديو (real-time-ish).
- `imgsz=640` قياسي، تقدر تزوده لو اللوحات في الفيديو بتاعك صغيرة جداً في الفريم (عربيات بعيدة عن الكاميرا).


In [ ]:
#@title تدريب Stage 1
from ultralytics import YOLO

model_stage1 = YOLO("yolo26s.pt")  # أوزان مدرّبة مسبقاً على COCO، هنعمل fine-tune على كشف اللوحات بس

results_stage1 = model_stage1.train(
    data=f"{dataset_stage1.location}/data.yaml",
    epochs=50,          # ابدأ بـ 50 وزوّد لو الدقة لسه بتتحسّن آخر epochs
    imgsz=640,
    batch=16,
    patience=15,        # early stopping لو مافيش تحسّن لـ 15 epoch
    project="alpr_runs",
    name="stage1_plate_detector",
)

STAGE1_WEIGHTS = "alpr_runs/stage1_plate_detector/weights/best.pt"
print("أفضل أوزان Stage 1:", STAGE1_WEIGHTS)


In [ ]:
#@title تقييم سريع لـ Stage 1 على val set
metrics_stage1 = model_stage1.val(data=f"{dataset_stage1.location}/data.yaml")
print("mAP50:", metrics_stage1.box.map50)
print("mAP50-95:", metrics_stage1.box.map)


## 4. تدريب Stage 2 — موديل كشف الحروف/الأرقام (YOLO26)

- 38 كلاس، والأجسام (الحروف) صغيرة جداً بالنسبة لصورة اللوحة المقصوصة — هنا بالظبط تحسينات YOLO26
  للأجسام الصغيرة (ProgLoss + STAL) بتفرق.
- ممكن تستخدم `yolo26m` بدل `yolo26s` لو عندك GPU قوي ووقت أطول — الدقة هتزيد شوية على حساب السرعة.


In [ ]:
#@title تدريب Stage 2
model_stage2 = YOLO("yolo26s.pt")

results_stage2 = model_stage2.train(
    data=f"{dataset_stage2.location}/data.yaml",
    epochs=80,           # عدد كلاسات أكتر ("38") محتاج epochs أكتر شوية من Stage 1
    imgsz=640,
    batch=16,
    patience=20,
    project="alpr_runs",
    name="stage2_char_detector",
)

STAGE2_WEIGHTS = "alpr_runs/stage2_char_detector/weights/best.pt"
print("أفضل أوزان Stage 2:", STAGE2_WEIGHTS)


In [ ]:
#@title تقييم سريع لـ Stage 2 على val set
metrics_stage2 = model_stage2.val(data=f"{dataset_stage2.location}/data.yaml")
print("mAP50:", metrics_stage2.box.map50)
print("mAP50-95:", metrics_stage2.box.map)
# لو mAP50 بعيد عن الـ 98.8% اللي حققها الموديل المرجعي على Roboflow، جرّب تزوّد epochs
# أو تشغّل augmentation إضافي (mosaic/mixup مفعّلين افتراضياً في ultralytics أصلاً)


## 5. تحويل أسماء الكلاسات (transliteration) لحروف عربية فعلية

أسماء الكلاسات في الداتاسِت مكتوبة بالحروف اللاتينية (transliteration) زي `jeem`, `7aa`, `Taa`... إلخ.
الجدول التالي هو تحويلها للحرف العربي الفعلي. **راجعه بعد ما تطبع `STAGE2_CLASS_NAMES` فوق** للتأكد إن
الأسماء مطابقة (لو فيه اختلاف بسيط في الإملاء، عدّل المفتاح المطابق في القاموس).


In [ ]:
#@title قاموس تحويل أسماء الكلاسات → حروف عربية
# ملحوظة: الفرق بين الحرف الصغير والكبير مقصود ومهم هنا (حسب تسمية الداتاسِت الأصلية):
#   taa (ت) مختلف عن Taa (ط)   |   thaa (ث) مختلف عن Thaa (ظ)
# و "7aa" بيمثّل حرف الحاء (ح) — استخدام الرقم 7 مكان الحاء ده تقليد شائع في الـ "Arabic chat alphabet"
# لأن مفيش حرف إنجليزي مكافئ للحاء.
CLASS_TO_ARABIC = {
    # الأرقام: زي ما هي (تقدر تحولها لأرقام هندية عربية ٠-٩ لو حابب بالقاموس اللي تحت)
    "0": "0", "1": "1", "2": "2", "3": "3", "4": "4",
    "5": "5", "6": "6", "7": "7", "8": "8", "9": "9",
    # الحروف
    "alif": "ا", "baa": "ب", "taa": "ت", "thaa": "ث", "jeem": "ج", "7aa": "ح",
    "khaa": "خ", "daal": "د", "zaal": "ذ", "raa": "ر", "zay": "ز", "seen": "س",
    "sheen": "ش", "saad": "ص", "daad": "ض", "Taa": "ط", "Thaa": "ظ", "ain": "ع",
    "ghayn": "غ", "faa": "ف", "qaaf": "ق", "kaaf": "ك", "laam": "ل", "meem": "م",
    "noon": "ن", "haa": "ه", "waw": "و", "yaa": "ي",
}

# رقم إنجليزي → رقم هندي-عربي (بيتظهر بيه على اللوحة فعلياً)، اختياري للعرض بس
WESTERN_TO_INDIC_DIGIT = dict(zip("0123456789", "٠١٢٣٤٥٦٧٨٩"))

# تأكيد إن كل كلاس في الداتاسِت ليه تحويل — لو طبعت تحذير هنا لازم تظبط القاموس فوق
missing = [c for c in STAGE2_CLASS_NAMES if c not in CLASS_TO_ARABIC]
if missing:
    print("⚠️ الكلاسات دي مش موجودة في القاموس، ظبّط CLASS_TO_ARABIC:", missing)
else:
    print("✅ كل الكلاسات (38) لها تحويل عربي صحيح")


## 6. دالة قراءة اللوحة: من صندوق مقصوص → نص عربي مرتّب

الفكرة: Stage 2 بيرجع كذا صندوق (حرف/رقم) لكل لوحة، من غير ترتيب. إحنا بنرتّبهم حسب مركز الصندوق أفقياً
(من الشمال لليمين زي ما هما فعلياً مطبوعين على اللوحة)، وبعدين نحوّل كل كلاس لحرفه العربي ونلزقهم في نص واحد.


In [ ]:
#@title decode_plate(): من نتيجة YOLO لصندوق مقصوص → نص اللوحة
import numpy as np

def decode_plate(stage2_result, conf_thresh=0.35):
    '''
    stage2_result: نتيجة واحدة من model_stage2.predict(plate_crop)[0]
    بترجع: (النص المرتّب, متوسط الثقة, عدد الحروف المكتشفة)
    '''
    boxes = stage2_result.boxes
    if boxes is None or len(boxes) == 0:
        return "", 0.0, 0

    xyxy = boxes.xyxy.cpu().numpy()      # (N, 4): x1,y1,x2,y2
    conf = boxes.conf.cpu().numpy()      # (N,)
    cls_ids = boxes.cls.cpu().numpy().astype(int)  # (N,)

    # فلترة الاكتشافات الضعيفة (احتمال إنها false positive)
    keep = conf >= conf_thresh
    xyxy, conf, cls_ids = xyxy[keep], conf[keep], cls_ids[keep]
    if len(xyxy) == 0:
        return "", 0.0, 0

    # مركز الصندوق أفقياً = نقطة الترتيب (من الشمال لليمين)
    x_centers = (xyxy[:, 0] + xyxy[:, 2]) / 2
    order = np.argsort(x_centers)

    chars = []
    for i in order:
        class_name = STAGE2_CLASS_NAMES[cls_ids[i]]
        chars.append(CLASS_TO_ARABIC.get(class_name, "?"))

    text = "".join(chars)
    avg_conf = float(conf.mean())
    return text, avg_conf, len(chars)


## 7. عرض النص العربي فوق فريم الفيديو

`cv2.putText` مايعرفش يرسم عربي (بيرسم الحروف منفصلة وبعكس الاتجاه). الحل القياسي: نشكّل النص بـ
`arabic_reshaper` + `python-bidi`، ونرسمه بـ PIL (اللي بتدعم TrueType fonts عربية) بدل OpenCV، وبعدين
نحوّل الصورة تاني لـ OpenCV array.


In [ ]:
#@title تحميل خط عربي وبناء دالة الرسم
import arabic_reshaper
from bidi.algorithm import get_display
from PIL import Image, ImageDraw, ImageFont
import cv2

# خط عربي مجاني من Google Fonts (Noto Naskh Arabic) — لو الرابط اتغيّر، دور على أي .ttf عربي وحطه بدله
!wget -q -O arabic_font.ttf "https://github.com/google/fonts/raw/main/ofl/notonaskharabic/NotoNaskhArabic-Regular.ttf"
ARABIC_FONT_PATH = "arabic_font.ttf"

def draw_arabic_text(frame_bgr, text, position, font_size=28, color=(0, 255, 0)):
    '''بترسم نص عربي (متشكّل واتجاهه صح) فوق فريم OpenCV (BGR) وترجع الفريم بعد الرسم.'''
    # 1) إعادة تشكيل الحروف (كل حرف بيتغيّر شكله حسب مكانه في الكلمة، زي الكتابة العادية)
    reshaped = arabic_reshaper.reshape(text)
    # 2) ضبط اتجاه الكتابة (RTL) — من غيرها الحروف بتتعرض بترتيب معكوس
    bidi_text = get_display(reshaped)

    # نحوّل الفريم لـ PIL عشان نرسم بيه فونط TrueType (OpenCV مايدعمش ده)
    img_pil = Image.fromarray(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB))
    draw = ImageDraw.Draw(img_pil)
    font = ImageFont.truetype(ARABIC_FONT_PATH, font_size)
    # PIL بياخد RGB مش BGR، فبنعكس ترتيب الألوان
    draw.text(position, bidi_text, font=font, fill=(color[2], color[1], color[0]))

    return cv2.cvtColor(np.array(img_pil), cv2.COLOR_RGB2BGR)


## 8. الـ Pipeline الكامل على فيديو

الخطوات لكل فريم:
1. `model_stage1.track()` بيكشف كل اللوحات في الفريم **ويدّي كل لوحة رقم تتبّع (track ID) ثابت** عبر
   الفريمات (مش detection مستقل كل مرة) — ده أساسي عشان نقدر "نصوّت" على قراءة نفس اللوحة عبر كذا فريم
   بدل ما نثق في قراءة فريم واحد بس.
2. نقصّ منطقة اللوحة (مع هامش بسيط) ونمررها لـ `model_stage2` عشان يكشف الحروف/الأرقام.
3. `decode_plate()` بترتّب الحروف وتطلع نص.
4. نجمع كل القراءات لكل track ID في `collections.Counter`، ونختار الأكتر تكراراً (majority vote) —
   ده بيصحّح تلقائياً أخطاء فريم واحد لو الفريمات التانية قرت اللوحة صح.
5. نرسم الصندوق + النص على الفريم ونكتبه في فيديو الإخراج.


In [ ]:
#@title تشغيل الـ pipeline كامل على فيديو
import collections

VIDEO_PATH = "input_video.mp4"   # TODO: حط مسار الفيديو بتاعك هنا (ارفعه لـ Colab أولاً)
OUTPUT_PATH = "output_annotated.mp4"
PLATE_CROP_PADDING = 5           # بكسل إضافي حوالين صندوق اللوحة قبل القص، بيحسّن كشف الحروف عالحواف
VOTE_MIN_READS = 3               # أقل عدد قراءات قبل ما نثق في نتيجة الـ majority vote

model_stage1 = YOLO(STAGE1_WEIGHTS)
model_stage2 = YOLO(STAGE2_WEIGHTS)

cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS) or 25
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
writer = cv2.VideoWriter(OUTPUT_PATH, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

# لكل track_id: عدّاد لكل نص متقروء (لعمل majority vote)، وآخر صندوق معروف (لو الكشف اتأخر فريم)
track_votes = collections.defaultdict(collections.Counter)

frame_idx = 0
while True:
    ok, frame = cap.read()
    if not ok:
        break
    frame_idx += 1

    # Stage 1: كشف + تتبّع اللوحات في الفريم ده (persist=True عشان يفتكر الـ track IDs بين الفريمات)
    track_results = model_stage1.track(frame, persist=True, verbose=False, tracker="bytetrack.yaml")[0]

    if track_results.boxes is not None and track_results.boxes.id is not None:
        boxes = track_results.boxes.xyxy.cpu().numpy()
        track_ids = track_results.boxes.id.cpu().numpy().astype(int)

        for box, tid in zip(boxes, track_ids):
            x1, y1, x2, y2 = box.astype(int)
            # نضيف هامش بسيط حوالين اللوحة قبل القص
            x1p, y1p = max(0, x1 - PLATE_CROP_PADDING), max(0, y1 - PLATE_CROP_PADDING)
            x2p, y2p = min(w, x2 + PLATE_CROP_PADDING), min(h, y2 + PLATE_CROP_PADDING)
            plate_crop = frame[y1p:y2p, x1p:x2p]
            if plate_crop.size == 0:
                continue

            # Stage 2: كشف الحروف/الأرقام جوه القصّة
            char_result = model_stage2.predict(plate_crop, verbose=False)[0]
            text, avg_conf, n_chars = decode_plate(char_result)

            if text:  # سجّل القراءة دي في تصويت اللوحة دي (track_id)
                track_votes[tid][text] += 1

            # القراءة النهائية المعروضة = الأكتر تكراراً لحد دلوقتي لنفس اللوحة (أكثر استقراراً من قراءة فريم واحد)
            if track_votes[tid]:
                best_text, votes = track_votes[tid].most_common(1)[0]
                label = best_text if votes >= 1 else "..."
            else:
                label = "..."

            # ارسم صندوق اللوحة
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
            # ارسم النص العربي فوق الصندوق (باستخدام دالة الرسم اللي عملناها فوق)
            frame = draw_arabic_text(frame, label, (x1, max(0, y1 - 35)), font_size=26)

    writer.write(frame)

    if frame_idx % 50 == 0:
        print(f"معالجة فريم {frame_idx}...")

cap.release()
writer.release()
print(f"✅ خلصنا. الفيديو الناتج: {OUTPUT_PATH}")

# ملخص القراءات النهائية (بعد التصويت) لكل لوحة اتتبّعت في الفيديو
print("\nملخص القراءات النهائية لكل لوحة:")
for tid, counter in track_votes.items():
    best_text, votes = counter.most_common(1)[0]
    total = sum(counter.values())
    print(f"  Track {tid}: \"{best_text}\"  (ظهرت في {votes}/{total} قراءة)")


In [ ]:
#@title (اختياري) عرض الفيديو الناتج جوه الـ notebook
from IPython.display import Video
Video(OUTPUT_PATH, embed=True, width=640)


## 9. تجربة سريعة على صورة واحدة (بدل فيديو كامل) — مفيدة للتشخيص السريع

نفس منطق الـ pipeline لكن على صورة ثابتة، مفيد وأنت بتظبط `conf_thresh` أو `CLASS_TO_ARABIC` وعايز تشوف
النتيجة فوراً من غير ما تشغّل فيديو كامل.


In [ ]:
#@title تجربة على صورة واحدة
import matplotlib.pyplot as plt

TEST_IMAGE_PATH = "test_car.jpg"  # TODO: صورة عربية فيها لوحة واضحة

img = cv2.imread(TEST_IMAGE_PATH)
plate_result = model_stage1.predict(img, verbose=False)[0]

fig, ax = plt.subplots(figsize=(10, 8))
if plate_result.boxes is not None and len(plate_result.boxes) > 0:
    box = plate_result.boxes.xyxy.cpu().numpy()[0].astype(int)
    x1, y1, x2, y2 = box
    crop = img[y1:y2, x1:x2]

    char_result = model_stage2.predict(crop, verbose=False)[0]
    text, avg_conf, n_chars = decode_plate(char_result)

    print(f"النص المقروء: {text}")
    print(f"عدد الحروف/الأرقام المكتشفة: {n_chars}")
    print(f"متوسط الثقة: {avg_conf:.2f}")

    annotated = draw_arabic_text(img.copy(), text, (x1, max(0, y1 - 35)), font_size=30)
    cv2.rectangle(annotated, (x1, y1), (x2, y2), (0, 255, 0), 2)
    ax.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
else:
    print("مافيش لوحة اتكشفت في الصورة دي")
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
ax.axis("off")
plt.show()


## 10. تقييم النظام + خطوات تحسين تالية

### مقاييس لازم تتابعها (مش mAP بس)
- **Character accuracy**: نسبة الحروف الصح من كل الحروف (edit distance / طول النص الصحيح).
- **Exact plate match**: نسبة اللوحات اللي اتقرت **كاملة** صح — ده المقياس اللي فعلياً بيهم في أي تطبيق حقيقي
  (بوابة/كاميرا مخالفات...إلخ)، لأن حرف واحد غلط بيخلي اللوحة كلها غلط عملياً.
- قيّم الاتنين **بعد التصويت (majority vote)** مش على قراءة فريم واحد، لأن ده الرقم اللي المستخدم النهائي هيشوفه.

### خطوات تحسين مرتّبة حسب الأولوية
1. **Perspective correction قبل Stage 2**: اللوحات في فيديو حقيقي بتكون مايلة بزاوية غالباً. لو ضفت
   تصحيح منظوري بسيط (4-point warp) بعد Stage 1 وقبل قص اللوحة، هتقلل أخطاء الحروف كتير في اللوحات المايلة.
2. **داتا Synthetic إضافية**: ولّد لوحات مصرية صناعية (خطوط حقيقية + إضاءة/ضبابية/غبار عشوائي) عشان تغطي
   حالات نادرة في الداتا الحالية (إضاءة ليلية، لوحات متسخة).
3. **فلترة بصيغة اللوحة المصرية**: لوحات مصر ليها نمط معروف (أرقام + حروف بترتيب معيّن) — ضيف regex أو
   قاعدة بسيطة بعد `decode_plate()` ترفض/تصحح نتائج مش متوافقة مع النمط ده.
2. **دقّة الـ tracker**: لو فيه لوحات بتضيع الـ track ID بتاعها (خصوصاً لو العربية اتغطّت لحظة)، جرّب
   `tracker="botsort.yaml"` بدل `bytetrack.yaml` — أدق شوية بس أبطأ.
5. **موديل أكبر لو الدقة مش كفاية**: `yolo26m`/`yolo26l` بدل `yolo26s` في الـ Stage اللي دقته أقل.
6. **Confusion matrix للحروف**: بعد التدريب، اعمل confusion matrix لـ Stage 2 على val set — الحروف
   المتشابهة بصرياً (زي ه/ة، ح/ج/خ) هتكون أكتر حاجة بتتلخبط، وتقدر تزوّد augmentation مستهدف ليها.
